Calculate source and OSD diversities for div steps over runs

In [ ]:
import pandas as pd
import numpy as np
import sys, os
sys.path.append(os.path.abspath(".."))
import config

In [ ]:
concat_feat_path = f"{config.path_to_data}/concat/CONCAT_feat.parquet"

In [ ]:
features = ['erraticness', 'intermittency', 'abs_energy',
       'agg_autocorrelation_max', 'agg_linear_trend_slope', 'kurtosis', 'mean',
       'median', 'skewness', 'standard_deviation'] 
       # , 'entropy',
       # 'seasonal_strength', 'hurst_exponent', 'DN_HistogramMode_5',
       # 'DN_HistogramMode_10', 'CO_f1ecac', 'CO_FirstMin_ac',
       # 'CO_HistogramAMI_even_2_5', 'CO_trev_1_num', 'MD_hrv_classic_pnn40',
       # 'SB_BinaryStats_mean_longstretch1',
       # 'SB_TransitionMatrix_3ac_sumdiagcov', 'PD_PeriodicityWang_th0_01',
       # 'CO_Embed2_Dist_tau_d_expfit_meandiff',
       # 'IN_AutoMutualInfoStats_40_gaussian_fmmi',
       # 'FC_LocalSimple_mean1_tauresrat', 'DN_OutlierInclude_p_001_mdrmd',
       # 'DN_OutlierInclude_n_001_mdrmd', 'SP_Summaries_welch_rect_area_5_1',
       # 'SB_BinaryStats_diff_longstretch0', 'SB_MotifThree_quantile_hh',
       # 'SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1',
       # 'SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1',
       # 'SP_Summaries_welch_rect_centroid', 'FC_LocalSimple_mean3_stderr']

In [ ]:
runs = ["run1", "run2", "run3", "run4", "run5"]

Average source diversity 

In [ ]:
results = {}

concat_feat = pd.read_parquet(concat_feat_path)[
            ["id_ts"] + features
        ]

for div_step in ["10", "5", "0"]:
    div_over_runs = []
    for run in runs:
            
        source = pd.read_parquet(f"{config.path_to_data}/{run}/interim/source_div{div_step}{run}/source_div{div_step}{run}.parquet")
        source_feat = concat_feat[concat_feat.id_ts.isin(source.id_ts.unique())]
        variance_ratio = source_feat.var(numeric_only=True) / concat_feat.var(
            numeric_only=True
        )
        sum_variance_ratios = variance_ratio.sum()
        div_over_runs.append(sum_variance_ratios)
    
    mean_sum = np.mean(div_over_runs)
    std_sum = np.std(div_over_runs)
    
    results[f'div_{div_step}'] = f"{mean_sum:.4f} ({std_sum:.4f})"

for div_name, result in results.items():
    print(f"\n{div_name}: {result}")
    

Average OSD diversity

In [ ]:
# OSD diversity
results = {}

concat_feat = pd.read_parquet(concat_feat_path)[[
            "id_ts"] + features]       

for div_step in ["10", "5", "0"]:
    div_over_runs = []
    for run in runs:
            
        source = pd.read_parquet(f"{config.path_to_data}/{run}/original_sources/percentile_{div_step}.parquet")
        source_feat = concat_feat[concat_feat.id_ts.isin(source.id_ts.unique())]
        variance_ratio = source_feat.var(numeric_only=True) / concat_feat.var(
            numeric_only=True
        )
        sum_variance_ratios = variance_ratio.sum()
        div_over_runs.append(sum_variance_ratios)
    
    mean_sum = np.mean(div_over_runs)
    std_sum = np.std(div_over_runs)
    
    results[f'div_{div_step}'] = f"{mean_sum:.4f} ({std_sum:.4f})"

for div_name, result in results.items():
    print(f"\n{div_name}: {result}")
    